# NMX

In [ ]:
import scipp as sc
import tof

import scippnexus as snx
from ess.reduce.unwrap import GenericUnwrapWorkflow
from ess.reduce.nexus.types import *
from ess.reduce.unwrap.types import *
from ess.reduce.unwrap.lut import LtotalRange, ChopperFrameSequence

source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)

## Chopper parameters

In [ ]:
choppers = {
    "chopper1": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-38.5], "unit": "deg"},
        "close": {"value": [38.5], "unit": "deg"},
        "distance": {"value": 28.4, "unit": "m"},
        "phase": {"value": 111.2, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "chopper1",
    },
    "chopper2a": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-70.0], "unit": "deg"},
        "close": {"value": [70.0], "unit": "deg"},
        "distance": {"value": 50.9774, "unit": "m"},
        "phase": {"value": 194.1, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "chopper2a",
    },
    "chopper2b": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-70.0], "unit": "deg"},
        "close": {"value": [70.0], "unit": "deg"},
        "distance": {"value": 51.0024, "unit": "m"},
        "phase": {"value": 168.0, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "chopper2b",
    },
}

In [ ]:
nmx_choppers = {}
for key, ch in choppers.items():
    nmx_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in nmx_choppers.items():
    print(key)
    display(ch)

## Tof model

In [ ]:
source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)
source_position = sc.vector([0, 0, 0], unit="m")
detector = tof.Detector(distance=sc.scalar(157.5, unit="m"), name="detector")

params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in nmx_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

## Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = nmx_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = sc.scalar(5, unit="m"), detector.distance
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot()

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()